# Week 4: Vector Power-Up & Raster Integration

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Master vector aggregation (dissolve & groupby)
2. Understand raster data as NumPy arrays with spatial metadata
3. Compute terrain slope from DEM
4. Extract raster values into vector shapes (zonal statistics)

**Packages needed:** `geopandas`, `rioxarray`, `rasterstats`, `numpy`, `matplotlib`

> If you haven't installed them yet, run:  
> `pip install rioxarray rasterstats geopandas folium mapclassify matplotlib`

## Cell [1] — Environment Setup

Import all necessary packages and load Week 3 township data.

In [2]:
# 導入所需套件
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import rioxarray as rxr  # 後續需要時再導入
# import xarray as xr      # 後續需要時再導入
from urllib.parse import quote

# 載入 TGOS 鄉鎮資料
TGOS_BASE = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/'
url = TGOS_BASE + quote('鄉(鎮、市、區)界線1140318.zip')

# 載入圖層
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')

# 轉換為 EPSG:3826
townships = townships.to_crs('EPSG:3826')

# 印出資訊
print(f"資料形狀: {townships.shape}")
print(f"座標系統: {townships.crs}")
print("\n前5行資料:")
print(townships.head())

資料形狀: (368, 8)
座標系統: EPSG:3826

前5行資料:
  TOWNID  TOWNCODE COUNTYNAME TOWNNAME             TOWNENG COUNTYID  \
0    V02  10014020        臺東縣      成功鎮  Chenggong Township        V   
1    T21  10013210        屏東縣      佳冬鄉    Jiadong Township        T   
2    P13  10009130        雲林縣      麥寮鄉    Mailiao Township        P   
3    V11  10014110        臺東縣      綠島鄉      Lüdao Township        V   
4    V16  10014160        臺東縣      蘭嶼鄉      Lanyu Township        V   

  COUNTYCODE                                           geometry  
0      10014  POLYGON ((286994.568 2569686.978, 287066.318 2...  
1      10013  POLYGON ((203528.677 2484633.419, 203536.121 2...  
2      10009  POLYGON ((178880.906 2634847.772, 178880.637 2...  
3      10014  MULTIPOLYGON (((300508.457 2508651.739, 300539...  
4      10014  MULTIPOLYGON (((313194.739 2427361.112, 313146...  


## Cell [2] — Dissolve & Groupby

**Dissolve**: Merge 368 townships → ~22 counties  
**Groupby**: Count shelters per county and sum capacity

In [5]:
# Cell [2]: YOUR CODE HERE
# 1. Dissolve townships by 'COUNTYNAME' → counties GeoDataFrame
# 2. Print: how many counties? What geometry types?
# 3. Calculate area in km² for each county
# 4. Show the top 5 largest counties

# AI Prompt suggestion:
# "Dissolve my townships GeoDataFrame by COUNTYNAME to create county boundaries.
#  Calculate each county's area in km². Show the top 5 largest."
# 將鄉鎮按 'COUNTYNAME' 進行 dissolve → 縣市 GeoDataFrame
counties = townships.dissolve(by='COUNTYNAME')
 
# 印出：有多少個縣市？幾何類型是什麼？
print(f"縣市數量: {len(counties)}")
print(f"幾何類型: {counties.geometry.type.unique()}")
 
# 計算每個縣市的面積（平方公里）
counties['area_km2'] = counties.geometry.area / 1_000_000  # 轉換為平方公里
 
# 顯示面積最大的前5個縣市
print("\n面積最大的前5個縣市:")
print(counties[['area_km2']].sort_values('area_km2', ascending=False).head())

縣市數量: 22
幾何類型: ['Polygon' 'MultiPolygon']

面積最大的前5個縣市:
               area_km2
COUNTYNAME             
花蓮縣         4605.284306
南投縣         4097.729914
臺東縣         3582.210917
高雄市         2998.488305
屏東縣         2805.036755


---

## 🔬 Lab 1: Vector Aggregation (20 minutes)

**Goal**: Practice dissolve & groupby before entering the raster world.

> **⚠️ 注意**：Lab 1 使用**合成資料**（synthetic data）練習 dissolve/groupby 操作。課後作業（ARIA v2.0）會使用**真實的避難收容所 CSV**，操作方式相同但資料來源不同。

### Step 1: Dissolve townships → counties

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# Dissolve all townships into counties using COUNTYNAME
# Verify: pick Hualien County (花蓮縣) and check its geometry type and area

# AI Prompt suggestion:
# "Dissolve my townships GeoDataFrame by 'COUNTYNAME'.
#  Then filter for '花蓮縣' and print its geometry type and area in km²."
# 將所有鄉鎮按 COUNTYNAME 進行 dissolve 來建立縣市
counties = townships.dissolve(by='COUNTYNAME')

# 驗證：選擇花蓮縣並檢查其幾何類型和面積
hualien = counties.loc['花蓮縣']
print(f"花蓮縣幾何類型: {hualien.geometry.type}")
print(f"花蓮縣面積: {hualien.geometry.area / 1_000_000:.2f} 平方公里")

### Step 2: Create synthetic shelter data for Hualien

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# Create synthetic shelter points in Hualien County
# Each town should have 2-8 shelters with random capacity (50-500)
# Place shelters at each town's centroid (simplified)

# AI Prompt suggestion:
# "Filter my townships for '花蓮縣' towns. For each town, create 2-8 random shelters
#  at the town centroid with random capacity 50-500.
#  Use np.random.seed(42) for reproducibility.
#  Create a GeoDataFrame with columns: shelter_id, TOWNNAME, capacity, geometry.
#  CRS should be EPSG:3826."


### Step 3: Groupby — Statistics per town

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# Group shelters by TOWNNAME
# Calculate: count, total capacity, average capacity
# Sort by total capacity descending

# AI Prompt suggestion:
# "Group my shelters GeoDataFrame by 'TOWNNAME'.
#  For each town, calculate shelter_count, total_capacity (sum), and avg_capacity (mean).
#  Sort by total_capacity descending and display all rows."


### Step 4: Merge stats back to geometry & visualize

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Merge the groupby statistics back to the Hualien township geometry
# 2. Create an interactive map with .explore() showing total_capacity by color
# 3. Save the map as HTML (for Windsurf/VS Code users)

# AI Prompt suggestion:
# "Merge my town_summary DataFrame back into the Hualien townships GeoDataFrame
#  using TOWNNAME. Then create an .explore() map colored by total_capacity
#  using 'YlOrRd' colormap. Add tooltip showing TOWNNAME, shelter_count, total_capacity.
#  Remember to convert to EPSG:4326 for .explore().
#  Also save the map as 'outputs/lab1_hualien_capacity.html' using m.save()."


---

## Cell [3] — The Raster Glass Box: It's Just a Matrix

A raster (like a DEM) is a **NumPy array with a GPS**. Each pixel = one numeric value.

In [ ]:
# Cell [3]: YOUR CODE HERE
# Create a synthetic 100x80 DEM with rioxarray
# Use valley profile: low center, mountains on sides
# Set CRS to EPSG:3826
# Print: shape, CRS, resolution, min/max elevation, top-left 5x5 values

# AI Prompt suggestion:
# "Create a synthetic 100x80 DEM simulating a valley (low center, high sides).
#  Elevation formula: 50 + 800*(2*X-1)^2 + 200*Y + 30*sin(8X)*cos(6Y)
#  where X and Y are normalized 0-1 meshgrids.
#  Wrap as xarray DataArray with rioxarray, set CRS EPSG:3826.
#  Use bounds: x=[302000, 303600], y=[2638000, 2640000].
#  Print shape, CRS, resolution, min/max, and the top-left 5x5 pixel values."


## Cell [4] — Affine Transform

How does `Matrix[row, col]` become real-world `(X, Y)` coordinates?

In [ ]:
# Cell [4]: YOUR CODE HERE
# Read the affine transform from your DEM
# Print the 6 parameters and explain what each means
# Manually convert pixel [5, 10] to real-world coordinates

# AI Prompt suggestion:
# "Get the affine transform from my DEM (dem.rio.transform()).
#  Print each of the 6 parameters with labels (pixel width, rotation, top-left coords).
#  Then manually compute the real-world coordinates of pixel [5, 10]
#  using: real_x = a*col + b*row + c, real_y = d*col + e*row + f.
#  Also print the elevation value at that pixel."


## Cell [5] — DEM Visualization

Create a side-by-side plot: elevation + hillshade.

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a 1x2 subplot: left = elevation with 'terrain' colormap, right = hillshade
# Save as outputs/cell5_dem.png

# AI Prompt suggestion:
# "Create a 1x2 matplotlib figure (14x5 inches).
#  Left: imshow of my DEM elevation with 'terrain' colormap and colorbar labeled 'Elevation (m)'.
#  Right: hillshade using matplotlib.colors.LightSource(azdeg=315, altdeg=45).
#  Use origin='upper'. Save as outputs/cell5_dem.png at 150 dpi."


---

## 🔬 Lab 2: Raster Exploration on Colab (15 minutes)

### ⚡ 請選擇你的環境：

| | **Option A: Google Colab（建議）** | **Option B: 本機 fallback** |
|---|---|---|
| 適用情境 | 網路正常、Drive mount 成功 | Colab 有問題或無法連線 |
| DEM 來源 | `dem_20m_hualien.tif`（Pre-lab 已下載到 Drive） | 使用 Cell [3] 的合成 DEM (`dem_synth`) |
| 後續 Step 2-4 | 用 `dem` 變數 | 用 `dem_synth` 變數（已在前面建立） |

> **Fallback 快速切換**：如果 Colab 不通，跳過 Step 1，直接到 Step 2，把 `dem` 改成 `dem_synth` 即可繼續。
> 
> **Note**: Steps 5-6 (zonal stats & risk classification) are included for reference — they will be completed in the homework.

### Step 1: Colab Setup

In [ ]:
# Lab 2 Step 1: Run in Colab!
# 1. Install packages: !pip install rioxarray rasterstats geopandas -q
# 2. Mount Google Drive
# 3. Import all packages

# AI Prompt suggestion:
# "Set up my Colab environment:
#  1. pip install rioxarray rasterstats geopandas
#  2. Mount Google Drive with drive.mount('/content/drive')
#  3. Import rioxarray, geopandas, numpy, matplotlib.pyplot, rasterstats"


### Step 2: Load DEM & Inspect

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# Load the pre-clipped Hualien DEM from Google Drive
# Print: shape, CRS, transform, bounds, elevation range, total pixels

# === Option A: Colab ===
# dem = rioxarray.open_rasterio('/content/drive/MyDrive/GIS_data/dem_20m_hualien.tif')

# === Option B: Local fallback ===
# dem = dem_synth  # 使用 Cell [3] 建立的合成 DEM

# AI Prompt suggestion:
# "Load a GeoTIFF DEM with rioxarray.open_rasterio().
#  Print its shape, CRS, transform, bounds, min/max elevation, and total pixel count."


### Step 3: Visualize DEM

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# Create a DEM elevation map with colorbar

# AI Prompt suggestion:
# "Visualize my DEM with matplotlib imshow using 'terrain' colormap.
#  Add colorbar labeled 'Elevation (m)'. Figure size 10x8."


### Step 4: Compute Slope

In [ ]:
# Lab 2 Step 4: YOUR CODE HERE
# Compute slope from DEM using np.gradient
# IMPORTANT: pixel spacing = 20 (for 20m DEM)
# Print: slope min, max, mean, % pixels > 30°
# Visualize: side-by-side elevation vs slope

# AI Prompt suggestion:
# "Compute slope from my DEM:
#  1. Extract 2D elevation array: dem.values[0]
#  2. Use np.gradient(elevation, 20) — 20 is the pixel spacing in meters
#  3. slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
#  4. Print min, max, mean slope and percentage of pixels > 30 degrees
#  5. Create 1x2 subplot: elevation (terrain cmap) vs slope (YlOrRd cmap)"


### Step 5: Zonal Statistics

In [ ]:
# Lab 2 Step 5: YOUR CODE HERE
# 1. Create shelter points within DEM extent (use synthetic data from earlier)
# 2. Create 200m buffers around shelters
# 3. CHECK CRS ALIGNMENT! (shelter CRS must match DEM CRS)
# 4. Run zonal_stats for elevation: mean, min, max, std
# 5. Run zonal_stats for slope: mean, max
# 6. Merge results back to shelter GeoDataFrame

# AI Prompt suggestion:
# "I have shelter points (EPSG:3826) and a DEM GeoTIFF.
#  1. First verify both are in the same CRS
#  2. Create 200m buffers around shelters
#  3. Save DEM and slope as temporary GeoTIFFs
#  4. Use rasterstats.zonal_stats() to get mean/min/max/std elevation per buffer
#  5. Use zonal_stats() to get mean/max slope per buffer
#  6. Merge results back and print a table with shelter_id, mean_elev, max_slope"


### Step 6: Risk Classification & Visualization

In [ ]:
# Lab 2 Step 6: YOUR CODE HERE
# 1. Classify terrain risk: HIGH (slope>30°), MEDIUM (20-30°), LOW (<20°)
# 2. Print risk summary
# 3. Create scatter plot: mean_elevation vs max_slope, colored by risk level
# 4. Save as outputs/lab2_terrain_risk.png

# AI Prompt suggestion:
# "Classify my shelters by terrain risk:
#  - HIGH: max_slope > 30°
#  - MEDIUM: max_slope 20-30°
#  - LOW: max_slope < 20°
#  Print the risk distribution.
#  Create a scatter plot of mean_elev vs max_slope, colored by risk (red/orange/green).
#  Add a horizontal line at slope=30° threshold.
#  Save as outputs/lab2_terrain_risk.png"


---

## Week 4 Reflection

Answer in the markdown cell below:

### My Reflection (fill in)

1. **Why is "distance to river" alone insufficient for flood risk assessment?**
   
   *Your answer:*

2. **What causes zonal_stats to return NaN, and how do you fix it?**
   
   *Your answer:*

3. **When combining vector and raster, should you reproject the vector or the raster? Why?**
   
   *Your answer:*

4. **What was the most surprising thing you learned about raster data today?**
   
   *Your answer:*